# 2.1 Understanding Embeddings

**What they are, how they work, and how to visualize them.**

In this notebook you will:
- Learn what embeddings are and why they matter for AI
- Convert text into numerical vectors using a pre-trained model
- Measure similarity between sentences using cosine similarity
- Visualize embeddings in 2D to see how meaning clusters together

> **No API key needed!** Everything in this notebook runs locally on your machine.

## 1. Setup

We install the libraries we need:
- **sentence-transformers**: converts text into embedding vectors
- **numpy**: numerical operations on arrays
- **matplotlib**: plotting and visualization
- **scikit-learn**: machine learning utilities (we'll use t-SNE and cosine similarity)

In [ ]:
# Install all required packages quietly (-q suppresses verbose output)
!pip install sentence-transformers numpy matplotlib scikit-learn -q

## 2. What is an Embedding?

An **embedding** is a way to represent text as a list of numbers (a **vector**).

Think of it like this:
- Every sentence gets a **coordinate in a high-dimensional space**
- Sentences with **similar meaning** end up **close together**
- Sentences with **different meaning** end up **far apart**

```
"The flight was delayed"  →  [0.12, -0.45, 0.78, ...] (384 numbers)
"The plane arrived late"  →  [0.11, -0.43, 0.80, ...] (very similar!)
"I love chocolate cake"   →  [-0.56, 0.23, 0.01, ...] (very different!)
```

This is the **foundation of semantic search** — instead of matching keywords, we match **meaning**.

## 3. Creating Embeddings

We'll use the `SentenceTransformer` library to load a pre-trained model and convert sentences into vectors.

In [ ]:
from sentence_transformers import SentenceTransformer

# SentenceTransformer loads a pre-trained model that converts text to vectors.
# "all-MiniLM-L6-v2" is a small (80MB), fast model good for learning.
# In production you'd use larger models for better quality.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Define three sentences — two are semantically similar, one is different
sentences = [
    "The flight from Istanbul to Tokyo was delayed 3 hours",
    "The plane arrived late to Narita airport",  # Similar meaning!
    "I ordered chocolate cake for dessert",       # Completely different
]

# .encode() converts each sentence into a vector of 384 numbers.
# The model reads the text, understands its meaning, and outputs a fixed-size vector.
embeddings = model.encode(sentences)

# Let's see the shape: (3 sentences, 384 dimensions per sentence)
print(f"Number of sentences: {embeddings.shape[0]}")
print(f"Each sentence becomes a vector of {embeddings.shape[1]} dimensions")
print(f"\nFirst 10 values of sentence 1: {embeddings[0][:10]}")

## 4. Measuring Similarity with Cosine Similarity

Now that we have vectors, how do we measure if two sentences are **similar**?

We use **cosine similarity**, which measures the angle between two vectors:
- **1.0** = identical meaning (vectors point the same direction)
- **0.0** = unrelated (vectors are perpendicular)
- **-1.0** = opposite meaning (vectors point opposite directions)

In practice, most text pairs score between 0.0 and 1.0.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# cosine_similarity compares every pair of vectors and returns a matrix.
# similarity_matrix[i][j] = how similar sentence i is to sentence j.
similarity_matrix = cosine_similarity(embeddings)

# Print a readable comparison of each pair
print("=== Cosine Similarity Between Sentences ===")
print()
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        score = similarity_matrix[i][j]
        print(f'Sentence {i+1}: "{sentences[i][:50]}..."')
        print(f'Sentence {j+1}: "{sentences[j][:50]}..."')
        print(f"Similarity:  {score:.4f}")
        # Interpret the score for the learner
        if score > 0.7:
            print("Verdict:     Very similar meaning!")
        elif score > 0.4:
            print("Verdict:     Somewhat related")
        else:
            print("Verdict:     Different topics")
        print()

Notice how the flight-related sentences score much higher than either does with the cake sentence. The model understood the **meaning**, not just the **words**.

## 5. Visualizing Embeddings with t-SNE

Each embedding has **384 dimensions** — way too many for humans to visualize. 

**t-SNE** (t-distributed Stochastic Neighbor Embedding) is a technique that **reduces** high-dimensional data to 2D or 3D while **preserving relationships**. Similar points stay close, different points stay far.

Let's embed more sentences and plot them to see clusters form!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Define sentences in 3 clear topic groups so we can see clustering
travel_sentences = [
    "The flight from Istanbul to Tokyo was delayed",
    "My plane arrived 3 hours late to the airport",
    "The airline cancelled our booking without notice",
    "We need to rebook the connecting flight",
]

hotel_sentences = [
    "The hotel room had a beautiful ocean view",
    "Check-in time is at 3 PM",
    "The resort offers complimentary breakfast",
    "Room service is available 24 hours",
]

food_sentences = [
    "I ordered chocolate cake for dessert",
    "The restaurant serves amazing pasta",
    "We tried local street food at the market",
    "The coffee shop has great espresso",
]

# Combine all sentences into one list
all_sentences = travel_sentences + hotel_sentences + food_sentences

# Create color labels so we can color-code the plot
# Each sentence gets a color based on its topic group
colors = ["#e74c3c"] * 4 + ["#3498db"] * 4 + ["#2ecc71"] * 4  # red, blue, green
labels = ["Flight"] * 4 + ["Hotel"] * 4 + ["Food"] * 4

# Encode all 12 sentences into 384-dimensional vectors
all_embeddings = model.encode(all_sentences)

In [ ]:
# t-SNE reduces 384 dimensions → 2 dimensions for plotting.
# perplexity controls how many neighbors each point considers (lower = tighter clusters).
# random_state makes the plot reproducible.
tsne = TSNE(
    n_components=2,      # Reduce to 2D
    perplexity=4,        # Small value because we only have 12 points
    random_state=42,     # For reproducibility
    n_iter=1000          # Number of optimization iterations
)

# Fit and transform: learn the 2D layout from the 384D embeddings
reduced = tsne.fit_transform(all_embeddings)

# Create the scatter plot
plt.figure(figsize=(10, 8))

# Plot each point with its color and label
for i, sentence in enumerate(all_sentences):
    plt.scatter(reduced[i, 0], reduced[i, 1], c=colors[i], s=100, zorder=5)
    # Annotate each point with a short version of the sentence
    short_text = sentence[:35] + "..." if len(sentence) > 35 else sentence
    plt.annotate(
        short_text,
        (reduced[i, 0], reduced[i, 1]),
        fontsize=8,
        ha="left",
        va="bottom",
        xytext=(5, 5),
        textcoords="offset points"
    )

# Add a legend explaining the colors
import matplotlib.patches as mpatches
legend_handles = [
    mpatches.Patch(color="#e74c3c", label="Flight/Travel"),
    mpatches.Patch(color="#3498db", label="Hotel"),
    mpatches.Patch(color="#2ecc71", label="Food"),
]
plt.legend(handles=legend_handles, loc="upper right", fontsize=11)

plt.title("Sentence Embeddings Visualized in 2D (t-SNE)", fontsize=14)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.tight_layout()
plt.show()

print("\nNotice how sentences about similar topics cluster together!")
print("This is the power of embeddings — they capture MEANING, not just keywords.")

## 6. Similarity Heatmap

Another way to visualize similarity is a **heatmap** — a grid where brighter colors mean higher similarity.

In [ ]:
# Calculate cosine similarity between ALL pairs of sentences
full_similarity = cosine_similarity(all_embeddings)

# Create short labels for the axes (first 30 chars of each sentence)
short_labels = [s[:30] + "..." for s in all_sentences]

# Plot the heatmap
plt.figure(figsize=(12, 10))
plt.imshow(full_similarity, cmap="YlOrRd", vmin=0, vmax=1)
plt.colorbar(label="Cosine Similarity")

# Add labels on both axes
plt.xticks(range(len(short_labels)), short_labels, rotation=45, ha="right", fontsize=7)
plt.yticks(range(len(short_labels)), short_labels, fontsize=7)

# Add similarity scores in each cell
for i in range(len(all_sentences)):
    for j in range(len(all_sentences)):
        plt.text(j, i, f"{full_similarity[i][j]:.2f}",
                 ha="center", va="center", fontsize=6,
                 color="white" if full_similarity[i][j] > 0.6 else "black")

plt.title("Cosine Similarity Heatmap — All Sentence Pairs", fontsize=13)
plt.tight_layout()
plt.show()

print("Bright blocks along the diagonal show within-topic similarity.")
print("The top-left 4x4 block (flights) and middle 4x4 block (hotels) should be brightest.")

## 7. YOUR TURN: Experiment!

Add your own sentences below and see where they cluster. Try:
- Adding sentences in different languages (the model handles multilingual text!)
- Adding ambiguous sentences (e.g., "I need to book a table" — travel or restaurant?)
- Adding sentences that use the same words but different meanings

In [ ]:
# YOUR TURN: Add your own sentences to this list!
# Then run this cell and the next one to see where they appear on the plot.
my_sentences = [
    "Replace this with your first sentence",
    "Replace this with your second sentence",
    "Replace this with your third sentence",
]

# Combine your sentences with the original ones
combined_sentences = all_sentences + my_sentences
combined_colors = colors + ["#9b59b6"] * len(my_sentences)  # Purple for your sentences

# Encode everything
combined_embeddings = model.encode(combined_sentences)

# Reduce to 2D with t-SNE
tsne_combined = TSNE(n_components=2, perplexity=4, random_state=42, n_iter=1000)
reduced_combined = tsne_combined.fit_transform(combined_embeddings)

# Plot
plt.figure(figsize=(10, 8))
for i, sentence in enumerate(combined_sentences):
    plt.scatter(reduced_combined[i, 0], reduced_combined[i, 1],
                c=combined_colors[i], s=100, zorder=5)
    short_text = sentence[:35] + "..." if len(sentence) > 35 else sentence
    plt.annotate(short_text, (reduced_combined[i, 0], reduced_combined[i, 1]),
                 fontsize=8, ha="left", va="bottom", xytext=(5, 5),
                 textcoords="offset points")

legend_handles = [
    mpatches.Patch(color="#e74c3c", label="Flight/Travel"),
    mpatches.Patch(color="#3498db", label="Hotel"),
    mpatches.Patch(color="#2ecc71", label="Food"),
    mpatches.Patch(color="#9b59b6", label="Your Sentences"),
]
plt.legend(handles=legend_handles, loc="upper right", fontsize=11)
plt.title("Where Do YOUR Sentences Cluster?", fontsize=14)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Embeddings** convert text into numerical vectors that capture meaning
2. **Similar sentences** get similar vectors (high cosine similarity)
3. **Pre-trained models** like `all-MiniLM-L6-v2` work out of the box — no training needed
4. **t-SNE** lets us visualize high-dimensional embeddings in 2D

**Next up:** In notebook **2.2**, we'll store these embeddings in a **vector database** and perform **semantic search**!